# NB6 · Web üzerinden çalışan arayüz

**Üretken Yapay Zekâ Araçları ile Klinik Karar Destek Sistemleri Geliştirilmesi**  
Sağlık Bilimlerinde Teknoloji ve Yapay Zekâ Okuryazarlığı Eğitimi · Akdeniz Üniversitesi · 18 Eylül 2026

Prof. Dr. Utku Köse · Süleyman Demirel Üniversitesi, Bilgisayar Mühendisliği Bölümü  
Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM) Müdürü · utkukose@sdu.edu.tr

---

Son defterde önceki adımlarda kurulan sistem tarayıcıdan kullanılabilen bir arayüze
bağlanır. Gradio, Colab içinde geçici bir genel bağlantı üretir.

Kontrol hücresi yalnızca defterin sonunda bulunmaktadır. Bu aşamada ürettiğiniz kodun
ne yaptığını okuyabilecek durumdasınız.


## Hazırlık


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
for modul in ['checks.py', 'evaluate.py', 'explain.py', 'safety.py',
              'mimic_web.py', 'pipeline.py']:
    urllib.request.urlretrieve(f'{REPO}/workshop/{modul}', modul)

import numpy as np
import pandas as pd
import checks, evaluate as ev, explain as ex, safety as sf

checks.LANG = ev.LANG = sf.LANG = 'tr'

!pip -q install gradio


In [ ]:
# Sabit hücre. Önceki defterlerin çıktısını yeniden kurar.
import pipeline as pl

durum = pl.prepare(verbose=False)
model = durum['model']
test = durum['test']
ozellikler = durum['features']
olasilik = durum['probabilities']
y_test = durum['y_test']
esik = ev.threshold_for_sensitivity(y_test, olasilik, target=0.80)
print(f'Model ve tahminler hazır. Çalışma eşiği: {esik:.3f}')


In [ ]:
# Sabit hücre. Bariyerli sistemi kurar.
bant = sf.choose_band(y_test, olasilik, esik, max_abstain=0.20)
korumali = sf.GuardedModel(
    model,
    sf.AbstentionPolicy(esik, bant['band']),
    sf.DriftDetector().fit(durum['train'][ozellikler]),
    sf.PhysiologicalValidator().fit(durum['train'][ozellikler]),
)

onem = ex.permutation_global(model, test[ozellikler], y_test, n_repeats=10, top=40)
sayisal = set(durum['train'][ozellikler].select_dtypes(include='number').columns)
KAYDIRICILAR = [f for f in onem['feature'] if f in sayisal][:6]
KATEGORIK = [f for f in ozellikler if f not in sayisal][:2]
VARSAYILAN = durum['train'][ozellikler].median(numeric_only=True)

print('Kaydırıcı:', KAYDIRICILAR)
print('Açılır liste:', KATEGORIK)


---

## Arayüzde neyin gösterileceği

Otuzdan fazla öznitelik için kaydırıcı koymak arayüzü kullanılamaz hâle getirir. En
önemli altı sayısal öznitelik seçilmiş, kalanlar eğitim kümesinin ortancasıyla
doldurulmuştur. Kullanıcı, görmediği özniteliklerin varsayılan değerde tutulduğunu bilmelidir.

Ekranda tek bir olasılık değil, kararın kendisi, bariyerlerin durumu ve katkı dökümü
birlikte yer alacaktır. Derste anlatılan ayrım buydu: Karar destek sisteminin ürünü bir
teşhis değil, gerekçesiyle birlikte sunulan bir dikkat tahsisidir.

Bir hususa ayrıca dikkat ediniz. Demografik öznitelikler klinik gerekçe olarak
sunulamaz. Katkı tablosunda göründüklerinde ayrı bir uyarı satırına taşınmalıdır; bu bir
adalet denetimi bulgusudur, klinisyene gösterilecek bir sebep değildir.


### İstem 1

```
Gradio ile tek sayfalık bir arayüz kuran Python hücresi yaz.

Elimde şunlar var:
  korumali      -> predict_one yöntemi tek satırlık DataFrame alıp sözlük döndürür;
                   sözlükte decision ve probability anahtarları bulunur
  KAYDIRICILAR  -> kaydırıcı konacak sayısal sütun adlarının listesi
  KATEGORIK     -> açılır liste konacak kategorik sütun adlarının listesi
  VARSAYILAN    -> sayısal sütunların ortanca değerleri
  ozellikler    -> modelin kullandığı bütün sütun adları
  durum['train'] -> eğitim kümesi DataFrame'i

Arayüz şöyle olsun:
1. Üstte başlık ve şu uyarı: Bu bir öğretim prototipidir, doğrulanmış bir klinik araç
   değildir; gerçek hasta kararlarında kullanılamaz.
2. Solda her KAYDIRICILAR sütunu için bir kaydırıcı, her KATEGORIK sütunu için bir
   açılır liste ve bir Değerlendir düğmesi.
3. Sağda kararın metin olarak gösterimi ve katkı tablosu.
4. Arayüzde gösterilmeyen öznitelik sayısını ve bunların ortanca değerde tutulduğunu
   ekranda belirt.
5. Katkı tablosunda gender, race, insurance veya marital_status geçen bir satır varsa
   onu tablodan çıkar ve bunun yerine ayrı bir adalet denetimi uyarısı göster.

Kaydırıcı sınırlarını eğitim kümesinin yüzde 1 ve yüzde 99 kuantillerinden al.

KABUL ÖLÇÜTLERİ
degerlendir adında bir fonksiyon üret. Kaydırıcı ve açılır liste değerlerini sırayla
alsın; iki değer döndürsün: Markdown metin ve bir DataFrame.
arayuz adında bir Gradio Blocks nesnesi üret.
Son satırda arayuz.launch(share=True) çağır.
```


In [ ]:
# Ürettiğiniz kodu bu hücreye yapıştırınız ve çalıştırınız.


---

## Defter sonu · Arayüz denetimi

Aşağıdaki hücre sabittir. Arayüzü açmadan fonksiyonu doğrudan çağırır ve üç durumda ne
döndürdüğünü gösterir.


In [ ]:
girdiler = ([float(VARSAYILAN.get(s, 0)) for s in KAYDIRICILAR] +
            [durum['train'][k].mode().iloc[0] for k in KATEGORIK])

metin, tablo = degerlendir(*girdiler)
print('OLAĞAN HASTA'); print(metin); print()

uc = [float(durum['train'][s].quantile(0.99)) * 40 for s in KAYDIRICILAR]
metin_uc, _ = degerlendir(*(uc + girdiler[len(KAYDIRICILAR):]))
print('AŞIRI DEĞERLİ HASTA'); print(metin_uc.split(chr(10))[0])


## Arayüzde denenecekler

Beş deneme yapınız.

Varsayılan değerlerle çalıştırınız ve kararın ne olduğuna bakınız. Ardından
kaydırıcıları yavaşça hareket ettirerek olasılığı eşiğe yaklaştırınız ve sistemin
çekimser kalmaya başladığı noktayı bulunuz. Klinik uygulamada bu bant, kimin listeye
alınacağını ve kimin klinisyene bırakılacağını belirler.

Birkaç kaydırıcıyı uçlara çekiniz. Sistem tahmin üretmeyi bırakıp devretme mesajı
vermelidir.

Farklı kaydırıcı birleşimleriyle benzer bir olasılığa ulaşınız ve katkı tablosunun
değiştiğini görünüz. İki hasta aynı riski taşıyabilir, ancak klinisyenin yapması gereken
farklıdır. Ekranda yalnızca bir sayı göstermenin neden yetersiz kaldığı burada görülür.

Son olarak cinsiyeti değiştirip başka hiçbir şeye dokunmayınız. Olasılık değişiyorsa ve
adalet denetimi uyarısı çıkıyorsa model demografik bir öznitelikten öğrenmiş demektir.
Bu bulgu, sistemin devreye alınmasını engelleyebilecek türdendir.


## Atölyenin sonu

Prototip çalışmaktadır ve bu, sistemin hazır olduğu anlamına gelmez. Derste anlatılan
zincirin yalnızca ilk halkaları tamamlanmıştır.

Eksik olanlar şunlardır: Başka bir merkezde dış doğrulama, klinisyen kaynaklı fizyolojik
sınırlar, iş akışına yerleştirme çalışması, eşiğin klinik ekipçe onaylanması, düzenleyici
sınıflandırma, kişisel veri değerlendirmesi ve devreye alma sonrası başarım izlemesi.

Bu liste, iki saatte yapılan işin klinik bir ürün yolculuğunun neresinde durduğunu
gösterir. Kod üretme maliyeti düşmüştür; geriye kalan hiçbir şey düşmemiştir.
---

**Uyarı.** Bu defterde üretilen hiçbir çıktı doğrulanmış bir klinik araç değildir.
MIMIC-IV demo verisi tek bir Amerikan hastanesinden gelmektedir ve Türkiye'deki bir
yoğun bakım popülasyonunu temsil etmez. Materyal öğretim amaçlıdır.
